# Tus funciones en Zenthros: aquí, en tu computadora y en una Spot

Este cuaderno es `tests/e2e/test_notebook_flow.py` hecho a mano:

1. Entrar con tu clave y ver **dónde puedes correr** (`zenthros.targets()`).
2. Subir tus datos al almacén del workspace (`ws.upload`).
3. Dos funciones normales, decoradas con lo que necesitan: una con Polars y scikit-learn, otra con Spark.
4. Correrlas aquí, en un contenedor de tu computadora y en una máquina Spot que se crea para la llamada y se apaga al terminar.
5. Una VM que se queda, una VM tuya unida con una línea, y borrar todo.

Necesitas:

- la clave `zen_…` que te da tu administrador (`zctl user add yo --workspace W --role editor --machines --provision`);
- Docker, para los contenedores de tu computadora;
- Python 3.13 con `pip install -e 'sdk/python[remote]' -r requirements.txt`, desde esta carpeta.

## 1. Entrar y ver dónde puedes correr

In [ ]:
import os
import sys
import tempfile

import zenthros

sys.path.insert(0, ".")  # esta carpeta: sus módulos viajan con cada llamada
import gen_data
import profiles

ws = zenthros.login()  # la primera vez: zenthros.login(key="zen_…"); después la recuerda
zenthros.targets()

`CPU` y `RAM` son lo que **tiene** una máquina de ese destino (o lo que tendría, para lo que aún no existe); nunca restan lo que está en uso. `AVAILABLE` es cuántas puedes usar o crear ahora. `zenthros.targets()["aws-spot"]` dice de dónde sale cada número.

## 2. Tus datos, al almacén del workspace

In [ ]:
# Fuera de esta carpeta: lo que hay en ella viaja con cada llamada.
local = os.path.join(tempfile.gettempdir(), "zenthros-ejemplo")
gen_data.generate(local, profiles.get("small"))  # 4,5 millones de líneas de pedido, 30 MB de Parquet
datos = ws.upload(local, to="ejemplo/datos")
salidas = f"{ws.store}/ejemplo/salidas"
datos

## 3. Dos funciones, con lo que necesitan

`requirements=` es un archivo junto al cuaderno; `tools=["java"]` pone Java donde corre. Zenthros prepara un entorno por función, una vez por máquina, y lo reutiliza.

In [ ]:
@zenthros.cloud(requirements="requirements-polars.txt", cpu=2, memory="4Gi")
def entrenar(datos, salida, perfil="small"):
    """Features con Polars y tres modelos de scikit-learn."""
    import pipeline

    return pipeline.models(datos, salida, perfil)


@zenthros.cloud(requirements="requirements-spark.txt", tools=["java"], cpu=2, memory="4Gi")
def tablas_spark(datos, salida, perfil="small"):
    """Las mismas tablas con Spark."""
    import pipeline

    return pipeline.tables(datos, salida, perfil)

## 4. Aquí, como cualquier función

In [ ]:
aqui = entrenar(local, os.path.join(local, "salidas", "aqui"))
{nombre: round(m["r2"], 3) for nombre, m in aqui["models"].items()}

## 5. En un contenedor de tu computadora

`create("local")` arranca un contenedor sin privilegios con límites duros (si la función se pasa, muere él y no tu equipo) y lo une a tu workspace, Privado. Devuelve la máquina cuando está lista. Con un `name` que ya existe, devuelve esa.

In [ ]:
mi_pc = zenthros.create("local", name="mi-pc", cpus=4, memory="8Gi")
zenthros.targets()

In [ ]:
resultado = entrenar.remote(on=mi_pc, params={"datos": datos, "salida": f"{salidas}/mi-pc/polars"})
print(resultado["seconds"], resultado["limits"])
spark = tablas_spark.remote(on=mi_pc, params={"datos": datos, "salida": f"{salidas}/mi-pc/spark"})
print(spark["rows"], spark["seconds"])

## 6. En una Spot, solo para esta llamada

Primero `dry_run=True`: qué se crearía, su CPU y su RAM frente a lo que pide la función, el precio, y si te falta algún permiso. No crea nada.

In [ ]:
entrenar.remote(on="aws-spot", params={"datos": datos, "salida": f"{salidas}/spot/polars"}, dry_run=True)

In [ ]:
# Crea una máquina del pool de la plantilla, corre, y el pool la apaga al quedar libre.
resultado = entrenar.remote(on="aws-spot", params={"datos": datos, "salida": f"{salidas}/spot/polars"})
print(resultado["seconds"], resultado["limits"])

## 7. Una VM que se queda, hasta que la borres

In [ ]:
vm = zenthros.create("aws-spot", name="vm-1")
print(zenthros.targets()["vm-1"])
zenthros.delete(vm)

## 8. Tu propia VM

`zenthros.attach("ubuntu@tu-host", key="~/.ssh/id_ed25519", name="mi-ec2")` entra por SSH y la une. Sin host, te da una línea para pegar en la VM (como un usuario con `sudo`). Queda **External**: Zenthros nunca la apaga, y `delete` solo la desune.

In [ ]:
comando = zenthros.attach(name="mi-ec2")
comando  # pégalo en la VM; después:
# mi_ec2 = comando.wait()
# zenthros.targets()["mi-ec2"]
# zenthros.delete(mi_ec2)

## 9. Borrar

In [ ]:
zenthros.delete(mi_pc)  # lo da de baja y borra el contenedor
zenthros.targets()